In [ ]:
import os
import pandas as pd

def consolidar_bases():
    # 1. Configurar os caminhos relativos para a pasta data/raw
    diretorio_raw = os.path.join("data", "raw")
    path_antigo = os.path.join(diretorio_raw, "fifa_rankings.csv")
    path_novo = os.path.join(diretorio_raw, "historico_ranking_fifa_corrigido.csv")
    path_saida = os.path.join(diretorio_raw, "fifa_consolidado.csv")

    print("Carregando os dataframes de data/raw/...")
    df_antigo = pd.read_csv(path_antigo)
    df_novo = pd.read_csv(path_novo)

    # 2. Padronizar o df_antigo (fifa_rankings.csv)
    # Garante que textos ou erros na pontuação antiga sejam forçados para nulo antes do ranking
    df_antigo['total_points'] = pd.to_numeric(df_antigo['total_points'], errors='coerce')
    df_antigo['rank'] = df_antigo.groupby('date')['total_points'].rank(ascending=False, method='min').astype("Int64")
    
    df_antigo = df_antigo.rename(columns={
        'date': 'date',
        'team': 'team',
        'total_points': 'points',
        'team_short': 'team_short'
    })
    df_antigo = df_antigo[['date', 'rank', 'team', 'points', 'team_short']]

    # 3. Padronizar o df_novo (historico_ranking_fifa_corrigido.csv)
    # Dicionário com o mapeamento oficial de datas
    mapa_datas = {
        "id14541": "2024-10-24",
        "id14576": "2024-11-28",
        "id14597": "2024-12-19",
        "id14702": "2025-04-03",
        "id14800": "2025-07-10",
        "id14870": "2025-09-18",
        "FRS_Male_Football_20250910": "2025-10-17",
        "FRS_Male_Football_20251015": "2025-11-19",
        "FRS_Male_Football_20251119": "2025-12-22",
        "FRS_Male_Football_20251219": "2026-01-19",
        "FRS_Male_Football_20260119": "2026-01-04",
        "LiveRanking": "2026-06-10"
    }

    # Substituir IDs pelas datas reais
    df_novo['date'] = df_novo['DateID'].map(mapa_datas).fillna(df_novo['DateID'])
    
    df_novo = df_novo.rename(columns={
        'Rank': 'rank',
        'Team': 'team',
        'Points': 'points'
    })
    
    # SOLUÇÃO DO ERRO: Transforma hifens "-" em nulos (NaN) e depois converte com segurança para Int64
    df_novo['rank'] = pd.to_numeric(df_novo['rank'], errors='coerce').astype("Int64")
    df_novo['points'] = pd.to_numeric(df_novo['points'], errors='coerce')
    
    df_novo = df_novo[['date', 'rank', 'team', 'points']]

    # 4. Juntar as duas bases verticalmente
    print("Concatenando e ordenando a base histórica completa...")
    df_completo = pd.concat([df_antigo, df_novo], ignore_index=True)

    # Remover possíveis sobreposições duplicadas de segurança e ordenar cronologicamente
    df_completo = df_completo.drop_duplicates(subset=['date', 'team'])
    df_completo = df_completo.sort_values(by=['date', 'rank'], ascending=[True, True])

    df_completo = df_novo.rename(columns={
        'points': 'total_points'
    })

    # 5. Salvar o arquivo final na pasta de destino
    os.makedirs(diretorio_raw, exist_ok=True)
    df_completo.to_csv(path_saida, index=False, encoding="utf-8")
    
    print(f"Sucesso! Base unificada salva em: {path_saida}")
    print(f"Dimensões finais da tabela: {df_completo.shape[0]} linhas por {df_completo.shape[1]} colunas.")

consolidar_bases()

Carregando os dataframes de data/raw/...
Concatenando e ordenando a base histórica completa...
Sucesso! Base unificada salva em: data\raw\fifa_consolidado.csv
Dimensões finais da tabela: 70372 linhas por 5 colunas.
